In [ ]:
# incremental_optdigits_profiles_ext.py
# Baseline vs Dual-Bullinaria vs Dual-Bullinaria-X2 (tablas grandes LaTeX)
# Ejecuta:  python incremental_optdigits_profiles_ext.py
# Requisitos: torch, scikit-learn, numpy, pandas, matplotlib

import math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, fetch_openml

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# =========================
# OPCIONES GLOBALES
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

dataset_source = "sklearn"          # "sklearn" (rápido) o "openml" (UCI completo)
reset_fast_between_sessions = True  # aplica sólo a Dual-Bullinaria-X2
SEEDS = tuple(range(5))             # para paper: subir a 30–100
PROFILES_TO_RUN = ["Baseline","Dual-Bullinaria","Dual-Bullinaria-X2"]


Dispositivo: cpu


In [ ]:
# =========================
# UTILIDADES
# =========================
def set_seed(seed: int):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


In [ ]:
# =========================
# DATOS Y LOTES
# =========================
def build_batches(dataset_source="sklearn", items_per_batch=200, n_batches=6):
    if dataset_source == "openml":
        Xy = fetch_openml('optdigits', version=1, as_frame=False)
        X = Xy.data.reshape(-1,8,8).astype('float32')/16.0
        y = Xy.target.astype('int64')
    else:
        d = load_digits()
        X = d.images.astype('float32')/16.0
        y = d.target.astype('int64')

    n_classes = 10
    per_class_per_batch = items_per_batch // n_classes  # 20
    rng = np.random.default_rng(42)

    per_class = {c: np.where(y==c)[0].tolist() for c in range(n_classes)}
    for c in per_class: rng.shuffle(per_class[c])

    batches, cursor = [], {c:0 for c in range(n_classes)}
    for _ in range(n_batches):
        idx = []
        for c in range(n_classes):
            s = cursor[c]; e = s + per_class_per_batch
            e = min(e, len(per_class[c]))
            idx.extend(per_class[c][s:e]); cursor[c] = e
        rng.shuffle(idx); batches.append(idx)

    used = set(sum(batches, []))
    rest_idx = [i for i in range(len(X)) if i not in used]
    rng.shuffle(rest_idx)
    split = int(0.6 * len(rest_idx))
    val_idx, test_idx = rest_idx[:split], rest_idx[split:]

    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)

    def make_loader(idx, batch_size=128, shuffle=False):
        ds = TensorDataset(X_tensor[idx], y_tensor[idx])
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

    batch_loaders_tpl = [(idx, make_loader(idx, 64, True)) for idx in batches]
    val_loader  = make_loader(val_idx, 128, False)
    test_loader = make_loader(test_idx,128, False)

    print("Tamaños B:", [len(b) for b in batches], "| Val:", len(val_idx), "| Test:", len(test_idx))
    return X_tensor, y_tensor, batch_loaders_tpl, val_loader, test_loader

In [ ]:
# =========================
# MODELOS
# =========================
class MLP(nn.Module):
    """Baseline: MLP 64->128->64->10 (capas registradas en __init__)."""
    def __init__(self, h1=128, h2=64):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64, h1)
        self.fc2 = nn.Linear(h1,  h2)
        self.fc3 = nn.Linear(h2, 10)
    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

class DualLinear(nn.Module):
    """Capa dual: pesos 'main' + 'fast' que se suman en el forward."""
    def __init__(self, in_f, out_f):
        super().__init__()
        self.w_main = nn.Parameter(torch.empty(out_f, in_f))
        self.b_main = nn.Parameter(torch.empty(out_f))
        self.w_fast = nn.Parameter(torch.zeros(out_f, in_f))
        self.b_fast = nn.Parameter(torch.zeros(out_f))
        nn.init.kaiming_uniform_(self.w_main, a=math.sqrt(5))
        bound = 1.0 / math.sqrt(in_f)
        nn.init.uniform_(self.b_main, -bound, bound)
    def forward(self, x):
        W = self.w_main + self.w_fast
        b = self.b_main + self.b_fast
        return x @ W.T + b

class MLPDual(nn.Module):
    """MLP con DualLinear en todas las capas densas."""
    def __init__(self, h1=128, h2=64):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = DualLinear(64, h1)
        self.fc2 = DualLinear(h1,  h2)
        self.fc3 = DualLinear(h2, 10)
    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


In [ ]:
# =========================
# ENTRENAMIENTO / EVAL
# =========================
criterion = nn.CrossEntropyLoss()

def train_epochs(model, loader, optimizer, epochs, device):
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

@torch.no_grad()
def accuracy(model, loader, device):
    model.eval(); total, correct = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += xb.size(0)
    return correct / total if total else float('nan')

In [ ]:
# =========================
# OPTIMIZADORES (perfiles)
# =========================
def build_optimizer_baseline(model, lr=1e-3):
    return torch.optim.Adam(model.parameters(), lr=lr)

def build_optimizer_dual(model, eta_main=1e-3, sigma=17.0, lambda_main=0.0,
                         delta_fast=1.1e-3, momentum=0.9, slow_IH_factor=1000.0):
    main_fc1    = [p for n,p in model.named_parameters() if 'fc1' in n and 'main' in n]
    main_rest   = [p for n,p in model.named_parameters() if ('fc2' in n or 'fc3' in n) and 'main' in n]
    fast_params = [p for n,p in model.named_parameters() if 'fast' in n]
    lr_fc1  = max(eta_main/slow_IH_factor, 1e-7)
    lr_fast = sigma*eta_main
    return torch.optim.SGD([
        {'params': main_fc1,   'lr': lr_fc1,  'weight_decay': lambda_main},
        {'params': main_rest,  'lr': eta_main,'weight_decay': lambda_main},
        {'params': fast_params,'lr': lr_fast, 'weight_decay': delta_fast},
    ], momentum=momentum)

PROFILES = {
    "Baseline": {
        "desc": "MLP estándar con Adam; sin fast weights.",
        "model": "mlp",
        "optimizer": lambda m: build_optimizer_baseline(m, lr=1e-3),
        "epochs_per_session": 200,
    },
    "Dual-Bullinaria": {
        "desc": "Dual: σ≈17, δ≈1.1e-3, λ≈0; IH muy lenta; 400 épocas.",
        "model": "dual",
        "optimizer": lambda m: build_optimizer_dual(
            m, eta_main=1e-3, sigma=17.0, lambda_main=0.0, delta_fast=1.1e-3,
            momentum=0.9, slow_IH_factor=1000.0),
        "epochs_per_session": 400,
    },
    "Dual-Bullinaria-X2": {
        "desc": "Dual intensificado: σ≈34, δ≈2.2e-3; reset fast opcional; 400 épocas.",
        "model": "dual",
        "optimizer": lambda m: build_optimizer_dual(
            m, eta_main=1e-3, sigma=34.0, lambda_main=0.0, delta_fast=2.2e-3,
            momentum=0.9, slow_IH_factor=1000.0),
        "epochs_per_session": 400,
    },
}

def reset_fast_params_(model):
    for n,p in model.named_parameters():
        if 'w_fast' in n or 'b_fast' in n:
            with torch.no_grad(): p.zero_()

In [ ]:
# =========================
# BUCLE INCREMENTAL
# =========================
def run_incremental(profile_name, X_tensor, y_tensor, batch_loaders_tpl, val_loader, test_loader,
                    seeds=SEEDS, hidden1=128, hidden2=64):
    cfg = PROFILES[profile_name]
    rows = []
    for seed in seeds:
        set_seed(seed)
        model = (MLP(hidden1,hidden2).to(device) if cfg['model']=='mlp' else MLPDual(hidden1,hidden2).to(device))
        optimizer = cfg['optimizer'](model)
        epochs_per_session = cfg['epochs_per_session']

        for t in range(6):
            idx_t, loader_t = batch_loaders_tpl[t]
            if reset_fast_between_sessions and profile_name == "Dual-Bullinaria-X2":
                reset_fast_params_(model)
            train_epochs(model, loader_t, optimizer, epochs_per_session, device)

            acc_Bt   = accuracy(model, loader_t, device)
            if t>0:
                prev_idx = np.concatenate([b for b,_ in batch_loaders_tpl[:t]]).tolist()
                prev_loader = DataLoader(TensorDataset(X_tensor[prev_idx], y_tensor[prev_idx]),
                                         batch_size=256, shuffle=False)
                acc_prev = accuracy(model, prev_loader, device)
            else:
                acc_prev = float('nan')
            acc_val  = accuracy(model, val_loader, device)
            acc_test = accuracy(model, test_loader, device)

            rows.append({'seed':seed, 'profile':profile_name, 'T':t+1,
                         'Acc_Bt':acc_Bt, 'Acc_prev':acc_prev, 'Acc_val':acc_val, 'Acc_test':acc_test})
    return pd.DataFrame(rows)

def mean_se(series):
    m = series.mean()
    se = series.std(ddof=1)/math.sqrt(len(series)) if len(series)>1 else 0.0
    return m, se

def summarize_table(df, profile_name):
    out=[]
    for T in range(1,7):
        sub = df[(df['profile']==profile_name) & (df['T']==T)]
        m_bt, se_bt   = mean_se(sub['Acc_Bt'])
        m_prev,se_prev= mean_se(sub['Acc_prev'].dropna())
        m_val, se_val = mean_se(sub['Acc_val'])
        m_test,se_test= mean_se(sub['Acc_test'])
        out.append({'T':T,
                    'Acc_Bt_mean':m_bt, 'Acc_Bt_se':se_bt,
                    'Acc_prev_mean':m_prev, 'Acc_prev_se':se_prev,
                    'Acc_val_mean':m_val, 'Acc_val_se':se_val,
                    'Acc_test_mean':m_test,'Acc_test_se':se_test})
    return pd.DataFrame(out)


In [ ]:
# =========================
# TABLAS LATEX GRANDES
# =========================
def pm(m, se):  # mean ± se en %
    return f"{m*100:.2f} $\\pm$ {se*100:.2f}"

def write_tables_wide_and_by_metric(summaries, profiles, prefix=""):
    # Tabla ancha conjunta: 4 métricas x P perfiles
    header_sections = []
    for metric in ["Bt","Prev","Val","Test"]:
        header_sections.append(f"\\multicolumn{{{len(profiles)}}}{{c}}{{{metric}}}")
    header_top = " & " + " & ".join(header_sections) + " \\\\"
    subheader = "T & " + " & ".join([p for _ in range(4) for p in profiles]) + " \\\\"

    lines = []
    lines.append("\\begin{tabular}{l" + "c"*(len(profiles)*4) + "}")
    lines.append("\\toprule")
    lines.append(header_top)
    # cmidrules de bloques
    b = len(profiles)
    lines.append(f"\\cmidrule(lr){{2-{1+b}}}\\cmidrule(lr){{{2+b}-{1+2*b}}}"
                 f"\\cmidrule(lr){{{2+2*b}-{1+3*b}}}\\cmidrule(lr){{{2+3*b}-{1+4*b}}}")
    lines.append(subheader)
    lines.append("\\midrule")

    for T in range(1,7):
        row = [str(T)]
        for pname in profiles:
            r = summaries[pname][summaries[pname]['T']==T].iloc[0]
            row.append(pm(r['Acc_Bt_mean'], r['Acc_Bt_se']))
        for pname in profiles:
            r = summaries[pname][summaries[pname]['T']==T].iloc[0]
            row.append(pm(r['Acc_prev_mean'], r['Acc_prev_se']))
        for pname in profiles:
            r = summaries[pname][summaries[pname]['T']==T].iloc[0]
            row.append(pm(r['Acc_val_mean'], r['Acc_val_se']))
        for pname in profiles:
            r = summaries[pname][summaries[pname]['T']==T].iloc[0]
            row.append(pm(r['Acc_test_mean'], r['Acc_test_se']))
        lines.append(" & ".join(row) + " \\\\")
    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    with open(f"{prefix}table_metrics_wide_profiles.tex","w") as f:
        f.write("\n".join(lines))
    print("Guardado:", f"{prefix}table_metrics_wide_profiles.tex")

    # Tablas por métrica
    def write_metric(metric_key, filename):
        map_mean = {'Bt':'Acc_Bt_mean','Prev':'Acc_prev_mean','Val':'Acc_val_mean','Test':'Acc_test_mean'}
        map_se   = {'Bt':'Acc_Bt_se',  'Prev':'Acc_prev_se',  'Val':'Acc_val_se',  'Test':'Acc_test_se'}
        L=[]
        L.append("\\begin{tabular}{l" + "c"*len(profiles) + "}")
        L.append("\\toprule")
        L.append("T & " + " & ".join(profiles) + " \\\\")
        L.append("\\midrule")
        for T in range(1,7):
            row=[str(T)]
            for pname in profiles:
                r = summaries[pname][summaries[pname]['T']==T].iloc[0]
                row.append(pm(r[map_mean[metric_key]], r[map_se[metric_key]]))
            L.append(" & ".join(row) + " \\\\")
        L.append("\\bottomrule")
        L.append("\\end{tabular}")
        with open(filename,"w") as f:
            f.write("\n".join(L))
        print("Guardado:", filename)

    write_metric('Bt',   f'{prefix}table_bt_profiles.tex')
    write_metric('Prev', f'{prefix}table_prev_profiles.tex')
    write_metric('Val',  f'{prefix}table_val_profiles.tex')
    write_metric('Test', f'{prefix}table_test_profiles.tex')

In [ ]:

# =========================
# MAIN
# =========================
if __name__ == "__main__":
    X_tensor, y_tensor, batch_loaders_tpl, val_loader, test_loader = build_batches(dataset_source)

    # Ejecutar perfiles
    all_results=[]
    for pname in PROFILES_TO_RUN:
        print(f"\nCorriendo perfil: {pname} — {PROFILES[pname]['desc']}")
        df = run_incremental(pname, X_tensor, y_tensor, batch_loaders_tpl, val_loader, test_loader, seeds=SEEDS)
        all_results.append(df)
    results = pd.concat(all_results, ignore_index=True)

    # CSV crudo
    results.to_csv("results_raw_incremental.csv", index=False)
    print("Guardado: results_raw_incremental.csv")

    # Resúmenes por perfil
    summaries={}
    for pname in PROFILES_TO_RUN:
        tbl = summarize_table(results, pname)
        summaries[pname]=tbl
        out_csv = f"summary_{pname.replace(' ','_')}.csv"
        tbl.to_csv(out_csv, index=False)
        print("Guardado:", out_csv)

    # Gráfica Acc_test (%)
    plt.figure()
    for pname in PROFILES_TO_RUN:
        tbl = summaries[pname]
        plt.plot(tbl['T'], tbl['Acc_test_mean']*100, marker='o', label=pname)
    plt.xlabel("Sesión T"); plt.ylabel("Test Accuracy (%)"); plt.title("Comparativa de perfiles")
    plt.ylim(0,100); plt.xlim(1,6); plt.legend(); plt.tight_layout()
    plt.savefig("plot_profiles_acc_test.png", dpi=150)
    print("Guardado: plot_profiles_acc_test.png")
    plt.close()

    # Gráfica Acc_prev (%)
    plt.figure()
    for pname in PROFILES_TO_RUN:
        tbl = summaries[pname]
        plt.plot(tbl['T'], tbl['Acc_prev_mean']*100, marker='s', label=pname)
    plt.xlabel("Sesión T"); plt.ylabel("Prev Accuracy (%)"); plt.title("Olvido (Prev) por T")
    plt.ylim(0,100); plt.xlim(1,6); plt.legend(); plt.tight_layout()
    plt.savefig("plot_profiles_acc_prev.png", dpi=150)
    print("Guardado: plot_profiles_acc_prev.png")
    plt.close()

    # Tablas LaTeX grandes
    write_tables_wide_and_by_metric(summaries, PROFILES_TO_RUN, prefix="")

    print("\nListo. Archivos generados en el directorio actual:\n"
          "- results_raw_incremental.csv\n"
          "- summary_*.csv\n"
          "- plot_profiles_acc_test.png, plot_profiles_acc_prev.png\n"
          "- table_metrics_wide_profiles.tex, table_bt_profiles.tex, table_prev_profiles.tex, table_val_profiles.tex, table_test_profiles.tex")

Tamaños B: [200, 200, 200, 200, 200, 200] | Val: 358 | Test: 239

Corriendo perfil: Baseline — MLP estándar con Adam; sin fast weights.

Corriendo perfil: Dual-Bullinaria — Dual: σ≈17, δ≈1.1e-3, λ≈0; IH muy lenta; 400 épocas.

Corriendo perfil: Dual-Bullinaria-X2 — Dual intensificado: σ≈34, δ≈2.2e-3; reset fast opcional; 400 épocas.
Guardado: results_raw_incremental.csv
Guardado: summary_Baseline.csv
Guardado: summary_Dual-Bullinaria.csv
Guardado: summary_Dual-Bullinaria-X2.csv
Guardado: plot_profiles_acc_test.png
Guardado: plot_profiles_acc_prev.png
Guardado: table_metrics_wide_profiles.tex
Guardado: table_bt_profiles.tex
Guardado: table_prev_profiles.tex
Guardado: table_val_profiles.tex
Guardado: table_test_profiles.tex

Listo. Archivos generados en el directorio actual:
- results_raw_incremental.csv
- summary_*.csv
- plot_profiles_acc_test.png, plot_profiles_acc_prev.png
- table_metrics_wide_profiles.tex, table_bt_profiles.tex, table_prev_profiles.tex, table_val_profiles.tex, table_